# 24.3 Evolution strategies (CMA-ES)

Evolution strategies sample real-valued candidates, keep elites, and adapt the search distribution.

Continuous black-box search uses a Gaussian around a mean instead of arbitrary encodings. The covariance matrix learns which directions successful elites occupy, so the optimizer can rotate and stretch its future samples.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

SEED = 243
rng = np.random.default_rng(SEED)


def sphere_loss(points, target=None):
    points = np.asarray(points, dtype=float)
    if target is None:
        target = np.zeros(points.shape[1])
    return np.sum(np.square(points - target), axis=1)


def ackley_loss(points):
    points = np.asarray(points, dtype=float)
    dim = points.shape[1]
    sum_sq = np.mean(np.square(points), axis=1)
    cos_term = np.mean(np.cos(2.0 * np.pi * points), axis=1)
    return -20.0 * np.exp(-0.2 * np.sqrt(sum_sq)) - np.exp(cos_term) + 20.0 + np.e


def constrained_ellipse_loss(points):
    raw = np.square(points[:, 0] / 2.0) + np.square(points[:, 1] * 2.0)
    violation = np.maximum(0.0, points[:, 0] + 0.5 * points[:, 1] - 1.0)
    return raw + 30.0 * np.square(violation)


def rosenbrock_rastrigin_loss(points):
    rosen = np.sum(100.0 * np.square(points[:, 1:] - np.square(points[:, :-1])) + np.square(1.0 - points[:, :-1]), axis=1)
    ras = 10.0 * points.shape[1] + np.sum(np.square(points) - 10.0 * np.cos(2.0 * np.pi * points), axis=1)
    return 0.1 * rosen + ras


def make_es_ladder():
    return [
        {"name": "D1 1-D quadratic bowl", "dim": 1, "bounds": (-5.0, 5.0), "loss": lambda x: np.square(x[:, 0] - 2.0)},
        {"name": "D2 2-D sphere", "dim": 2, "bounds": (-5.0, 5.0), "loss": lambda x: sphere_loss(x, target=np.array([1.0, 1.0]))},
        {"name": "D3 Ackley multimodal", "dim": 2, "bounds": (-5.0, 5.0), "loss": ackley_loss},
        {"name": "D4 constrained ellipse", "dim": 2, "bounds": (-4.0, 4.0), "loss": constrained_ellipse_loss},
        {"name": "D5 20-D Rosenbrock-Rastrigin", "dim": 20, "bounds": (-3.0, 3.0), "loss": rosenbrock_rastrigin_loss},
    ]


def weighted_covariance(elites, mean, weights, jitter=1e-6):
    centered = elites - mean
    cov = np.zeros((elites.shape[1], elites.shape[1]))
    for index in range(elites.shape[0]):
        cov = cov + weights[index] * np.outer(centered[index], centered[index])
    cov = cov + jitter * np.eye(elites.shape[1])
    return cov


def evolution_strategy(loss, dim, bounds, generations=45, population_size=16, elite_count=6, sigma=0.8, jitter=1e-6, min_variance=1e-4, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    lower, upper = bounds
    mean = rng.uniform(lower, upper, size=dim)
    covariance = np.eye(dim)
    history = []
    last_samples = None
    for generation in range(generations):
        covariance = covariance + jitter * np.eye(dim)
        samples = rng.multivariate_normal(mean, sigma * sigma * covariance, size=population_size)
        samples = np.clip(samples, lower, upper)
        values = loss(samples)
        order = np.argsort(values)
        elites = samples[order[:elite_count]]
        weights = np.log(elite_count + 0.5) - np.log(np.arange(1, elite_count + 1))
        weights = weights / np.sum(weights)
        mean = np.sum(elites * weights[:, None], axis=0)
        covariance = weighted_covariance(elites, mean, weights, jitter=jitter)
        diagonal = np.maximum(np.diag(covariance), min_variance)
        covariance = covariance.copy()
        np.fill_diagonal(covariance, diagonal)
        sigma = max(0.95 * sigma, 0.05)
        history.append(float(np.min(values)))
        last_samples = samples
    final_value = float(loss(mean.reshape(1, -1))[0])
    return {"mean": mean, "covariance": covariance, "best_loss": final_value, "history": np.array(history), "population": last_samples}


## The concept, built once (D1): elite recombination

The lesson samples from

$$x_k\sim\mathcal{N}(m_t,\sigma_t^2 C_t),\qquad m_{t+1}=\sum_{k=1}^{\mu}w_k x_{k:\lambda}.$$

For samples $(1,0),(0,1),(2,2),(-1,0)$ and target $(1,1)$, the top two elites are $(1,0)$ and $(0,1)$, so the new mean is $(0.5,0.5)$.

In [ ]:

samples = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 2.0], [-1.0, 0.0]])
target = np.array([1.0, 1.0])
distances = np.sum(np.square(samples - target), axis=1)
elites = samples[np.argsort(distances)[:2]]
new_mean = np.mean(elites, axis=0)

print("squared distances:", distances)
print("new mean:", new_mean)

assert np.allclose(distances, np.array([1.0, 1.0, 2.0, 5.0]))
assert np.allclose(new_mean, np.array([0.5, 0.5]))


## Covariance from elites

Centering the two elites around $(0.5,0.5)$ gives biased covariance entries $0.25$ on the diagonal and $-0.25$ off diagonal. A tiny $10^{-6}$ jitter is added during optimization so too few elites cannot make sampling singular.

In [ ]:

centered = elites - new_mean
covariance = centered.T @ centered / 2.0

print("elite covariance:\n", covariance)

assert np.isclose(covariance[0, 0], 0.25)
assert np.isclose(covariance[1, 1], 0.25)
assert np.isclose(covariance[0, 1], -0.25)
assert np.isclose(covariance[1, 0], -0.25)


## The dataset ladder

The inline ladder starts with a 1-D quadratic bowl, then uses a 2-D sphere, Ackley multimodality, a constrained ellipse, and a 20-D Rosenbrock-Rastrigin hybrid.

In [ ]:

ladder = make_es_ladder()

for rung in ladder:
    preview_rng = np.random.default_rng(SEED + rung["dim"])
    lower, upper = rung["bounds"]
    sample = preview_rng.uniform(lower, upper, size=(4, rung["dim"]))
    values = rung["loss"](sample)
    print(rung["name"], "shape", sample.shape, "sample loss", np.round(values, 3))


In [ ]:

results = []

for index, rung in enumerate(ladder):
    result = evolution_strategy(rung["loss"], rung["dim"], rung["bounds"], generations=45, population_size=18, elite_count=6, sigma=0.9, jitter=1e-6, rng=np.random.default_rng(SEED + index))
    results.append(result)
    print(f"{rung['name']}: best_loss={result['best_loss']:.4f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for index, rung in enumerate(ladder):
    ax = axes[0, index]
    lower, upper = rung["bounds"]
    if rung["dim"] == 1:
        grid = np.linspace(lower, upper, 200)
        ax.plot(grid, rung["loss"](grid.reshape(-1, 1)))
        ax.scatter(results[index]["mean"][0], results[index]["best_loss"], c="red")
    else:
        grid_x = np.linspace(lower, upper, 80)
        grid_y = np.linspace(lower, upper, 80)
        mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)
        points = np.column_stack([mesh_x.ravel(), mesh_y.ravel()])
        if rung["dim"] > 2:
            padded = np.zeros((points.shape[0], rung["dim"]))
            padded[:, :2] = points
            values = rung["loss"](padded)
            population = results[index]["population"][:, :2]
        else:
            values = rung["loss"](points)
            population = results[index]["population"]
        ax.contourf(mesh_x, mesh_y, values.reshape(mesh_x.shape), levels=20, cmap="magma_r")
        ax.scatter(population[:, 0], population[:, 1], c="white", s=10, edgecolor="black")
        if rung["dim"] == 2:
            cov = results[index]["covariance"]
            vals, vecs = np.linalg.eigh(cov)
            angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
            ellipse = Ellipse(results[index]["mean"], width=2.0 * np.sqrt(vals[-1]), height=2.0 * np.sqrt(vals[0]), angle=angle, fill=False, color="cyan")
            ax.add_patch(ellipse)
    ax.set_title(rung["name"].split()[0])

for index, result in enumerate(results):
    axes[1, index].plot(result["history"])
    axes[1, index].set_title("best loss")
    axes[1, index].set_xlabel("generation")

fig.suptitle("Sample clouds with covariance and best-loss curves")
plt.tight_layout()


## Pitfall on D5: forgetting covariance regularization

A tiny elite set can make the covariance singular or overconfident. The fixed run adds diagonal jitter and a minimum variance floor.

In [ ]:

d5 = ladder[-1]

fragile = evolution_strategy(d5["loss"], d5["dim"], d5["bounds"], generations=35, population_size=8, elite_count=2, sigma=0.6, jitter=0.0, min_variance=0.0, rng=np.random.default_rng(777))
stable = evolution_strategy(d5["loss"], d5["dim"], d5["bounds"], generations=35, population_size=8, elite_count=2, sigma=0.6, jitter=1e-6, min_variance=1e-4, rng=np.random.default_rng(777))

print("fragile min variance", np.min(np.diag(fragile["covariance"])), "loss", fragile["best_loss"])
print("regularized min variance", np.min(np.diag(stable["covariance"])), "loss", stable["best_loss"])


## Evaluate it + Practice

- Compare the reported best loss against a seeded random-search baseline with the same evaluation budget.
- Sanity check: D1 must hit the lesson's worked calculation before trusting D2–D5.
- Ablation: remove covariance jitter and the minimum variance floor should make the hardest rung worse or less stable.
- Failure signals: population variance collapses too early, bounds are violated, or the summary curve improves only by one lucky sample.
- Re-run with another seed only after the structural checks pass; do not tune from a single trace.

Practice prompts:
1. Change the hardest rung budget and explain whether convergence improves per objective call.
2. Add one diagnostic for diversity and plot it next to the metric.
3. Swap the D3 multimodal objective and predict which operator needs retuning.